<a href="https://colab.research.google.com/github/bipasnadulal/aws-future-ai-programmer-nanodegree/blob/main/training_a_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from pathlib import Path

text = Path("/content/tiny-shakespeare.txt").read_text()

In [ ]:
print(text[0:1000])

In [ ]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [ ]:
tokenizer = CharTokenizer.train_from_text(text)

In [ ]:
from torch.utils.data import Dataset


class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        assert pos < len(self.data) - self.block_size

        x = self.data[pos : pos + self.block_size]
        y = self.data[pos + 1 : pos + 1 + self.block_size]
        return x, y

In [ ]:
config = {
    "vocabulary_size": tokenizer.vocabulary_size(),
    "context_size": 256,
    "embedding_dim": 768,
    "heads_num": 12,
    "layers_num": 10,
    "dropout_rate": 0.1,
    "use_bias": False,
}

config["head_size"] = config["embedding_dim"] // config["heads_num"]

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.Q_weights = nn.Linear(
            config["embedding_dim"], config["head_size"], config["use_bias"]
        )
        self.K_weights = nn.Linear(
            config["embedding_dim"], config["head_size"], config["use_bias"]
        )
        self.V_weights = nn.Linear(
            config["embedding_dim"], config["head_size"], config["use_bias"]
        )

        self.dropout = nn.Dropout(config["dropout_rate"])

        casual_attention_mask = torch.tril(
            torch.ones(config["context_size"], config["context_size"])
        )
        self.register_buffer("casual_attention_mask", casual_attention_mask)

    def forward(self, input):
        batch_size, tokens_num, embedding_dim = input.shape
        Q = self.Q_weights(input)
        K = self.K_weights(input)
        V = self.V_weights(input)

        attention_scores = Q @ K.transpose(1, 2)
        attention_scores = attention_scores.masked_fill(
            self.casual_attention_mask[:tokens_num, :tokens_num] == 0, -torch.inf
        )
        attention_scores = attention_scores / (K.shape[-1] ** 0.5)
        attention_scores = torch.softmax(attention_scores, dim=-1)
        attention_scores = self.dropout(attention_scores)

        return attention_scores @ V

In [ ]:
input = torch.rand(8, config["context_size"], config["embedding_dim"])

In [ ]:
ah = AttentionHead(config)

output = ah(input)

output.shape

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        heads_list = [AttentionHead(config) for _ in range(config["heads_num"])]
        self.heads = nn.ModuleList(heads_list)

        self.linear = nn.Linear(config["embedding_dim"], config["embedding_dim"])
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input):
        # print(f"Input shape: {input.shape}")
        heads_outputs = [head(input) for head in self.heads]

        scores_change = torch.cat(heads_outputs, dim=-1)
        # print(f"heads shape: {scores_change.shape}")

        scores_change = self.linear(scores_change)
        return self.dropout(scores_change)


In [ ]:
mha = MultiHeadAttention(config)

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.linear_layers = nn.Sequential(
            nn.Linear(config["embedding_dim"], config["embedding_dim"] * 4),
            nn.GELU(),
            nn.Linear(config["embedding_dim"] * 4, config["embedding_dim"]),
            nn.Dropout(config["dropout_rate"]),
        )

    def forward(self, input):
        return self.linear_layers(input)

In [ ]:
ff = FeedForward(config)

In [ ]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.multi_head = MultiHeadAttention(config)
        self.layer_norm_1 = nn.LayerNorm(config["embedding_dim"])

        self.feed_forward = FeedForward(config)
        self.layer_norm_2 = nn.LayerNorm(config["embedding_dim"])

    def forward(self, input):
        residual = input
        x = self.multi_head(self.layer_norm_1(input))
        x = x + residual

        residual = x
        x = self.feed_forward(self.layer_norm_2(x))
        return x + residual

In [ ]:
class DemoGPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding_layer = nn.Embedding(
            config["vocabulary_size"], config["embedding_dim"]
        )
        self.positional_embedding_layer = nn.Embedding(
            config["context_size"], config["embedding_dim"]
        )

        blocks = [Block(config) for _ in range(config["layers_num"])]
        self.layers = nn.Sequential(*blocks)

        self.layer_norm = nn.LayerNorm(config["embedding_dim"])
        self.unembedding = nn.Linear(
            config["embedding_dim"], config["vocabulary_size"], bias=False
        )

    def forward(self, token_ids):
        # print("Forward")
        batch_size, tokens_num = token_ids.shape

        x = self.token_embedding_layer(token_ids)
        sequence = torch.arange(tokens_num, device=device)
        x = x + self.positional_embedding_layer(sequence)

        x = self.layers(x)
        x = self.layer_norm(x)
        x = self.unembedding(x)

        return x

In [ ]:
model = DemoGPT(config).to(device)

In [ ]:
def generate(model, prompt_ids, max_tokens):
    output_ids = prompt_ids
    for _ in range(max_tokens):
        if output_ids.shape[1] >= config["context_size"]:
            break
        with torch.no_grad():
            logits = model(output_ids)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        # Sample a random token given the softmax distribution
        next_token_id = torch.multinomial(probs, num_samples=1)
        # Add new token to the output, and repeat the process
        output_ids = torch.cat([output_ids, next_token_id], dim=-1)
    return output_ids

In [ ]:
def generate_with_prompt(model, tokenizer, prompt, max_tokens=100):
    model.eval()

    prompt = tokenizer.encode(prompt).unsqueeze(dim=0).to(device)

    return tokenizer.decode(generate(model, prompt, max_tokens=max_tokens)[0])

### Training a model

In [ ]:
batch_size = 64

train_iterations = 5000
evaluation_interval = 100
learning_rate = 4e-4

In [ ]:
train_data = tokenizer.encode(text).to(device)

In [ ]:
train_dataset = TokenIdsDataset(train_data, config['context_size'])

In [ ]:
from torch.utils.data import Dataset, DataLoader, RandomSampler

train_sampler = RandomSampler(
    train_dataset, num_samples=batch_size * train_iterations, replacement = True
)
train_dataloader = DataLoader(
    train_dataset, batch_size = batch_size, sampler = train_sampler
)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [29]:
for step_num, sample in enumerate(train_dataloader):
  model.train()
  input, targets = sample
  logits = model(input)

  logits_view = logits.view(
      batch_size * config['context_size'], config['vocabulary_size']
  )

  targets_view = targets.view(batch_size * config['context_size'])

  loss = F.cross_entropy(logits_view, targets_view)

  #backward propagation
  loss.backward()

  #update model parameters
  optimizer.step()

  #set to none to reduce memory usage
  optimizer.zero_grad(set_to_none=True)

  print(f"Step {step_num}. Loss {loss.item():.3f}")

  if step_num % evaluation_interval == 0:
    print('Demo GPT: \n' + generate_with_prompt(model, tokenizer, "\n"))


Step 0. Loss 4.351
Demo GPT: 

r  d   C              e o   i                     f i        Y   he   l    n       r   o            
Step 1. Loss 5.483
Step 2. Loss 4.327
Step 3. Loss 4.409
Step 4. Loss 3.969
Step 5. Loss 3.369
Step 6. Loss 3.125
Step 7. Loss 2.895
Step 8. Loss 2.902
Step 9. Loss 2.838
Step 10. Loss 2.778
Step 11. Loss 2.775
Step 12. Loss 2.790
Step 13. Loss 2.724
Step 14. Loss 2.702
Step 15. Loss 2.695
Step 16. Loss 2.681
Step 17. Loss 2.679
Step 18. Loss 2.625
Step 19. Loss 2.623
Step 20. Loss 2.658
Step 21. Loss 2.623
Step 22. Loss 2.588
Step 23. Loss 2.599
Step 24. Loss 2.567
Step 25. Loss 2.575
Step 26. Loss 2.541
Step 27. Loss 2.550
Step 28. Loss 2.534
Step 29. Loss 2.541
Step 30. Loss 2.510
Step 31. Loss 2.545
Step 32. Loss 2.528
Step 33. Loss 2.522
Step 34. Loss 2.512
Step 35. Loss 2.519
Step 36. Loss 2.526
Step 37. Loss 2.521
Step 38. Loss 2.522
Step 39. Loss 2.500
Step 40. Loss 2.524
Step 41. Loss 2.489
Step 42. Loss 2.495
Step 43. Loss 2.490
Step 44. Loss 2.4

KeyboardInterrupt: 